In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('/content/Sales_Final from SQL.csv')
df

# **Cleaning**

In [ ]:
# Data Cleaning
df['Product'] = df['Product'].str.title().str.strip()

df['Category'] = df['Category'].str.title().str.strip()

df.drop_duplicates(inplace=True)

In [ ]:
# Data Transformation and adding Columns and fillings

df['Price'] = pd.to_numeric(df['Price'] , errors = 'coerce')

df['Price'] = round(df['Price'].fillna(df.groupby('Product')['Price'].transform('mean')),2)

df['Revenue'] = round((df['Price'] * df['Quantity']),2)

def parse_mixed_dates(date):
    try:
         return pd.to_datetime(date, format='%Y-%m-%d')
    except:
        try:
            return pd.to_datetime(date, format='%m/%d/%Y')
        except:
            return pd.NaT

df['Date'] = df['Date'].apply(parse_mixed_dates)

df['Year']=df['Date'].dt.year
df['Month']=df['Date'].dt.month
df['Week Day']=df['Date'].dt.day_name()

In [ ]:
# Exploratory Data Analysis
Total_Sales = df['Revenue'].sum()
Total_Orders = df['OrderID'].nunique()
Average_Order_Value = round(Total_Sales / Total_Orders,2)
print(f'Total sales is = {Total_Sales}')
print(f'Total Orders is = {Total_Orders}')
print(f'Average Order Value is = {Average_Order_Value}')

Highest_product_rev =df.sort_values(by='Revenue',ascending=False,inplace=False)
Highest_product_rev
# Chicken

Highest_product_Vol =df.sort_values(by=['Quantity','Revenue'],ascending=[False,False],inplace=False)
Highest_product_Vol
# Meat

In [ ]:
# Branch Table is bf & Product Table is pf
bf = pd.read_csv('/content/branches.csv')
pf = pd.read_csv('/content/products.csv')

# **Brand** **Performance**

In [ ]:
# Tables Merging
df_merge_pf = pd.merge(df , pf, on = 'ProductID' , how="inner")

In [ ]:
df_merge_pf

In [ ]:
Brand_Performance = df_merge_pf.groupby('Brand')['Quantity'].sum()
Top_Brands= Brand_Performance.sort_values(  ascending= False)
Top_Brands
# Americana

In [ ]:
Brand_Leading_Categories = df_merge_pf.groupby('Category_y')['Brand'].value_counts()
Brand_Leading_Categories.sort_values(ascending=False)
Brand_Leading_Categories
# Juhayna in Dairy & Americana in Frozen

# **Location Analysis**

In [ ]:
# Tabels Merging
df_merge_bf = pd.merge(df , bf , on = 'BranchID' , how='inner')

In [ ]:
df_merge_bf

In [ ]:
City_Performance = df_merge_bf.groupby('City')['Revenue'].sum()
City_Performance.sort_values(ascending = False)
City_Performance
# Cairo

In [ ]:
#Which branches underperform compared to others in the same city?
Branch_Performance = df_merge_bf.groupby(['BranchID','City'])['Revenue'].sum()
Branch_Performance.sort_values(ascending = True)
# The lowest Branch is CAI-NAS-02	Cairo

In [ ]:
#What are the monthly revenue trends?
Monthly_Revenue_Trends = df_merge_bf.groupby('Month')['Revenue'].sum()
Monthly_Revenue_Trends
# May is the highest month in sales

In [ ]:
#Which days of the week have the highest sales?
Days_Performance = df_merge_bf.groupby('Week Day')['Revenue'].sum()
Days_Performance.sort_values(ascending = False)
# Wednesday Highest day in sales

In [ ]:
# Merging all tables toghther
Final_df_Cleaned = df.merge(pf, on='ProductID', how='inner') \
             .merge(bf, on='BranchID', how='inner')
Final_df_Cleaned

In [ ]:
Final_df_Cleaned.rename(columns={'Category_x' : 'Category' , 'Category_y' : 'Brand Category'} , inplace=True)
Final_df_Cleaned

In [ ]:
Final_df_Cleaned['Customer']= Final_df_Cleaned['Customer'].fillna('Unknown Customer')
Final_df_Cleaned

# **Visualization**

In [ ]:
# Revenue by Category

categories = Final_df_Cleaned['Category']
revenue = Final_df_Cleaned['Revenue']

plt.figure(figsize=(10, 6))
plt.bar(categories, revenue, color='teal')

plt.title('Total Revenue by Category', fontsize=14)
plt.xlabel('Category', fontsize=12)
plt.ylabel('Revenue (EGP)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Revenue Per Month

monthly_sales = Final_df_Cleaned.groupby('Month')['Revenue'].sum().reset_index()

monthly_sales = monthly_sales.sort_values('Month')

plt.figure(figsize=(10, 5))
plt.plot(monthly_sales['Month'], monthly_sales['Revenue'], color='teal', marker='o', linewidth=2)

plt.xlabel('Month Number')
plt.ylabel('Total Revenue (EGP)')
plt.title('Total Revenue Trend per Month')
plt.grid(True, linestyle='--', alpha=0.6)

plt.show()



In [ ]:
# Revenue by Brand
brand_revenue = Final_df_Cleaned.groupby('Brand')['Revenue'].sum()

plt.figure(figsize=(10, 8))
plt.pie(brand_revenue,
        labels=brand_revenue.index,
        autopct='%1.1f%%',
        startangle=140,
        colors=plt.cm.Paired.colors)

plt.title('Revenue Share by Brand', fontsize=15)
plt.axis('equal')

plt.show()

In [ ]:
# Revenue and Quantity by City
city_stats = Final_df_Cleaned.groupby('City')[['Revenue', 'Quantity']].sum().reset_index()
city_melted = city_stats.melt(id_vars='City', var_name='Metric', value_name='Value')
plt.figure(figsize=(12, 6))

sns.barplot(data=city_melted, x='City', y='Value', hue='Metric', palette='muted')

plt.title('Revenue vs Quantity per City', fontsize=15)
plt.xlabel('City', fontsize=12)
plt.ylabel('Total Value', fontsize=12)
plt.legend(title='Metric')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

In [ ]:
# Saving Final Table
Final_df_Cleaned.to_csv('Cleaned_Data (Python).csv', index=False)